In [1]:
# Lab 1 — Local Ollama
# Original additions and modifications © 2026 Dr Elena Kramer, Braude College, Israel.
# Adapted course content and third-party materials remain with their respective rights holders.

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [2]:
MODEL = "gpt-oss:120b"
OLLAMA_BASE_URL = "http://192.168.1.26:11434/v1"
if not OLLAMA_BASE_URL:
    raise ValueError("Set OLLAMA_BASE_URL in the project-root .env file")

openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [3]:
question_prompt = "Generate one technical interview question about Python programming. Respond only with the question, without an answer."
messages = [{"role": "user", "content": question_prompt}]
response = openai.chat.completions.create(model=MODEL, messages=messages)
question = response.choices[0].message.content
print(question)

How would you implement a thread-safe Least Recently Used (LRU) cache in Python without using the built-in `functools.lru_cache`, and what data structures and synchronization primitives would you choose to ensure O(1) get and put operations?


In [4]:
short_prompt = f"""
Answer the following Python interview question
in no more than two sentences:

{question}
"""

short_response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": short_prompt}
    ]
)

short_answer = short_response.choices[0].message.content

print(short_answer)

Implement the cache as a fixed‑capacity container whose entries are stored in a hash map that points to nodes in a doubly‑linked list ordered from most‑ to least‑recently used, and guard every `get` or `put` with a `threading.Lock` (or `RLock`) so that the node can be moved to the head or the tail evicted in constant time. This combination of a dict + linked list and a simple lock yields O(1) average‑case get/put operations while providing thread safety without using `functools.lru_cache`.


In [5]:
detailed_prompt = f"""
Answer the following Python interview question
in detail. Include a clear explanation and a code example:

{question}
"""

detailed_response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": detailed_prompt}
    ]
)

detailed_answer = detailed_response.choices[0].message.content

print(detailed_answer)

## 1.  What an LRU cache has to do  

| Operation | Desired time‑complexity |
|-----------|------------------------|
| **`get(key)`** – return the cached value and mark the entry as “most‑recently used” | **O(1)** |
| **`put(key, value)`** – insert a new entry (or update an existing one) and, if the cache is full, evict the *least* recently used entry | **O(1)** |

In addition, the implementation must be **thread‑safe**: several Python threads may call `get` and `put` concurrently and must never corrupt the internal state.

---

## 2.  Core data structures that give O(1) behaviour  

| Structure | Why it is needed |
|-----------|------------------|
| **Hash table (`dict`)** mapping *key → node* | Direct lookup of a node in constant time. |
| **Doubly‑linked list** that stores the nodes in *usage order* (head = most‑recent, tail = least‑recent) | Moving a node to the front or removing the tail node are both O(1) because we have direct `prev`/`next` pointers. |
| **Sentinel head / tail n

In [6]:
judge_prompt = f"""
You are a judge evaluating two answers to a Python interview question.

Question:
{question}

Answer 1:
{short_answer}

Answer 2:
{detailed_answer}

Compare the answers based on:
1. Correctness
2. Clarity
3. Completeness

Choose the better answer and briefly explain why.
"""

judge_response = openai.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": judge_prompt}
    ]
)

judge_result = judge_response.choices[0].message.content

print(judge_result)

**Chosen answer: Answer 2**

**Why it’s better**

| Criterion | Answer 1 | Answer 2 |
|-----------|----------|----------|
| **Correctness** | States the right approach (dict + doubly‑linked list + a lock) and that operations are O(1). | Gives a full, working implementation that correctly handles look‑ups, insertions, eviction, and thread safety. |
| **Clarity** | Very brief; the idea is there but a reader has to fill in many details (how the list is managed, edge cases, lock scope). | Structured with sections, tables, and explanatory comments. Each step (data structures, lock choice, helper methods) is spelled out, making it easy to follow. |
| **Completeness** | Only mentions the high‑level design; no code, no discussion of edge cases, no complexity proof, no examples. | Covers the full interview checklist: required O(1) operations, thread‑safety rationale, exact data structures, a complete Python class, a small multithreaded demo, complexity recap, and optional extensions. |

Thus **